# Hybrid Genetic Algorithm-Neural Network Modeling for Spray Drying of Coconut Milk
**Author:** @VedantAndhale

## Step 1: Generate Synthetic Data
We first create a synthetic dataset that mimics the spray drying process, including input parameters and resulting powder properties.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Generate synthetic dataset
np.random.seed(42)
N = 200
inlet_temp = np.random.uniform(150, 200, N)
feed_rate = np.random.uniform(0.5, 1.5, N)
moisture = 0.1 * inlet_temp + 2 * feed_rate + np.random.normal(0, 2, N)
particle_size = 0.05 * inlet_temp**1.2 + 1.5 * feed_rate + np.random.normal(0, 1.5, N)

df = pd.DataFrame(
    {
        "inlet_temp": inlet_temp,
        "feed_rate": feed_rate,
        "moisture": moisture,
        "particle_size": particle_size,
    }
)

X = df[["inlet_temp", "feed_rate"]].values
y = df[["moisture", "particle_size"]].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print("Synthetic data prepared:", df.shape)
display(df.head())

Synthetic data prepared: (200, 4)


,inlet_temp,feed_rate,moisture,particle_size
0,168.727006,1.142032,21.767722,24.376243
1,197.535715,0.584140,20.963859,30.437196
2,186.599697,0.661629,21.347133,28.293733
3,179.932924,1.398554,20.169867,26.047239
4,157.800932,1.106429,18.641284,23.520725


## Data Dictionary
- **inlet_temp**: Inlet air temperature during spray drying (°C)
- **feed_rate**: Feed solution flow rate into the dryer (L/min)
- **moisture**: Moisture content of the resulting coconut milk powder (% by weight)
- **particle_size**: Average particle diameter of the dried powder (µm)

Input features (`inlet_temp`, `feed_rate`) are used to predict output targets (`moisture`, `particle_size`).

## Step 2: Define Neural Network Model and Fitness Function
We define a simple feedforward neural network to predict powder properties, and a fitness function (mean squared error) for the genetic algorithm to optimize.

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import mean_squared_error
import random
import os

# Force TensorFlow to use CPU only
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"


# Build a neural network given hyperparams
def build_model(params):
    lr, hidden_units = params
    model = models.Sequential(
        [
            tf.keras.Input(shape=(2,)),
            layers.Dense(int(hidden_units), activation="relu"),
            layers.Dense(2),
        ]
    )
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=lr), loss="mse")
    return model


# Fitness: lower MSE -> higher fitness
def fitness(params):
    tf.keras.backend.clear_session()
    model = build_model(params)
    model.fit(X_train, y_train, epochs=20, batch_size=16, verbose=0)
    preds = model.predict(X_test, verbose=0)
    mse = mean_squared_error(y_test, preds)
    tf.keras.backend.clear_session()
    return -mse  # GA maximizes fitness

2025-04-22 19:10:49.717226: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-22 19:10:49.719093: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-22 19:10:49.727246: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-04-22 19:10:49.752195: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745329249.790808  208239 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745329249.80

## Step 3: Genetic Algorithm Optimization
We use the DEAP library to optimize the neural network's learning rate and hidden layer size using a genetic algorithm.

In [3]:
from deap import base, creator, tools


if "FitnessMax" not in creator.__dict__:
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if "Individual" not in creator.__dict__:
    creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Parameters: [learning_rate (0.0001-0.01), hidden_units (5-100)]
toolbox.register("lr", random.uniform, 1e-4, 1e-2)
toolbox.register("hu", random.randint, 5, 100)
toolbox.register(
    "individual", tools.initCycle, creator.Individual, (toolbox.lr, toolbox.hu), n=1
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", fitness)
toolbox.register("mate", tools.cxBlend, alpha=0.5)


# mutate learning rate and hidden units separately
def mutate_individual(individual, indpb):
    if random.random() < indpb:
        individual[0] = random.uniform(1e-4, 1e-2)
    if random.random() < indpb:
        individual[1] = random.randint(5, 100)
    return (individual,)


toolbox.register("mutate", mutate_individual, indpb=0.2)
toolbox.register("select", tools.selTournament, tournsize=3)

pop = toolbox.population(n=10)
NGEN = 5
for gen in range(NGEN):
    offspring = toolbox.select(pop, len(pop))
    offspring = list(map(toolbox.clone, offspring))
    for child1, child2 in zip(offspring[::2], offspring[1::2]):
        if random.random() < 0.5:
            toolbox.mate(child1, child2)
        toolbox.mutate(child1)
        toolbox.mutate(child2)
        del child1.fitness.values, child2.fitness.values
    invalid = [ind for ind in offspring if not ind.fitness.valid]
    for ind in invalid:
        ind.fitness.values = (toolbox.evaluate(ind),)
    pop = offspring

best = tools.selBest(pop, k=1)[0]

2025-04-22 19:10:53.480477: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2025-04-22 19:10:53.480537: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:167] env: CUDA_VISIBLE_DEVICES="-1"
2025-04-22 19:10:53.480549: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:170] CUDA_VISIBLE_DEVICES is set to -1 - this hides all GPUs from CUDA
2025-04-22 19:10:53.480557: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:178] verbose logging is disabled. Rerun with verbose logging (usually --v=1 or --vmodule=cuda_diagnostics=1) to get more diagnostic output from this module
2025-04-22 19:10:53.480566: I external/local_xla/xla/stream_executor/cuda/cuda_diagnostics.cc:183] retrieving CUDA diagnostic information for host: pop-os
2025-04-22 19:10:53.480571: I external/local_xla/xla/stream_executor/cuda/cuda_d

## Step 4: Results and Best Hyperparameters
Display the best hyperparameters found by the genetic algorithm for the neural network.

In [4]:
def_x = best  # ensure best is defined
print("Best hyperparameters (lr, hidden_units):", best)

Best hyperparameters (lr, hidden_units): [0.00498962341540021, 31]
